# TF-IDF Analysis for Transcript Words

Analyzes vocabulary patterns across transcripts using TF-IDF (Term Frequency-Inverse Document Frequency). This helps identify:
- Words that are distinctive to specific recordings
- Common vs. rare words in the dataset
- Word importance for classification tasks (Dementia, MCI, HC)

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.stem import WordNetLemmatizer

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support
)
from sklearn.utils.sparsefuncs import mean_variance_axis
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

nltk.download('wordnet', quiet=True)

import optuna
import dataloader

## Data Loading and Preprocessing

In [ ]:
loader = dataloader.TranscriptLoader()
transcript_paths = loader.discover_transcript_files()

frames_dict = loader.load_all_transcripts(transcript_paths)
metadata = loader.load_metadata()

lemmatizer = WordNetLemmatizer()

def lemma_tokenizer(text):
    tokens = re.split(r'\W+', text.lower())
    return [lemmatizer.lemmatize(w) for w in tokens if len(w) > 1]

diagnosis_by_recording = {
    row['recording']: row['diagnosis']
    for _, row in metadata[['recording', 'diagnosis']].dropna().iterrows()
}

TARGET_TASKS = ('SFT', 'PFT')
# TARGET_DIAGNOSES = ('DEMENTIA', 'MCI', 'HC')
TARGET_DIAGNOSES = ('DEMENTIA', 'HC')

task_diagnosis_frames = {
    task: {diagnosis: {} for diagnosis in TARGET_DIAGNOSES}
    for task in TARGET_TASKS
}

for stem, df in frames_dict.items():
    if len(df) == 0: continue

    if 'task' in df.columns:
        task = str(df['task'].iloc[0]).upper()
    else:
        task = 'SFT' if 'SFT' in stem.upper() else 'PFT' if 'PFT' in stem.upper() else 'UNKNOWN'

    recording = str(df['recording'].iloc[0]) if 'recording' in df.columns else None
    diagnosis = diagnosis_by_recording.get(recording, 'UNKNOWN')

    if task in TARGET_TASKS and diagnosis in TARGET_DIAGNOSES:
        task_diagnosis_frames[task][diagnosis][stem] = df

## Utility Functions for TF-IDF and Visualization

In [ ]:
def build_corpus(dataframes, min_tokens=2):
    corpus = []
    for df in dataframes:
        tokens = [t.strip() for t in df['token'].astype(str).tolist() if t.strip()]
        if len(tokens) < min_tokens: continue
        text = ' '.join(df['token'].astype(str).tolist())
        corpus.append(text)
    return corpus

def plot_tfidf_comparison(task_name, task_data, n_limit=20):
    fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=False)
    fig.suptitle(f"Top {n_limit} Stemmed Terms: {task_name} Task", fontsize=18, fontweight='bold')
    
    colors = {'HC': '#2ecc71', 'MCI': '#f1c40f', 'DEMENTIA': '#e74c3c'}

    for idx, diagnosis in enumerate(['HC', 'MCI', 'DEMENTIA']):
        ax = axes[idx]
        diagnosis_frames = task_data.get(diagnosis, {})
        corpus = build_corpus(diagnosis_frames.values())

        if not corpus:
            ax.text(0.5, 0.5, "No Data Available", ha='center', va='center')
            ax.set_title(f"{diagnosis}")
            continue

        vectorizer = TfidfVectorizer(tokenizer=lemma_tokenizer, token_pattern=None)
        X = vectorizer.fit_transform(corpus)
        feature_names = vectorizer.get_feature_names_out()
        mean_scores = mean_variance_axis(X, axis=0)[0]
        
        sorted_indices = mean_scores.argsort()[::-1][:n_limit]
        top_terms = [feature_names[i] for i in sorted_indices]
        top_values = [mean_scores[i] for i in sorted_indices]

        ax.barh(top_terms, top_values, color=colors.get(diagnosis, 'skyblue'))
        ax.set_title(f"{diagnosis} (n={len(corpus)})", fontsize=14)
        ax.invert_yaxis()
        ax.set_xlabel("Mean TF-IDF Score")
        ax.grid(axis='x', linestyle='--', alpha=0.7)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## Machine Learning Pipeline

In [ ]:


def build_tuned_pipeline(trial):
    """
    This replaces build_pipeline for the optimization phase.
    It uses trial.suggest to 'try' different values.
    """
    # 1. Hyperparameters for TF-IDF
    max_feat = trial.suggest_int('tfidf__max_features', 10, 200)
    binary_val = trial.suggest_categorical('tfidf__binary', [True, False])
    
    # 2. Hyperparameters for Logistic Regression
    c_val = trial.suggest_float('clf__C', 1e-3, 10.0, log=True)
    
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            tokenizer=lemma_tokenizer,
            token_pattern=None,
            max_features=max_feat,
            binary=binary_val,
            sublinear_tf=True
        )),
        ('clf', LogisticRegression(
            class_weight='balanced', 
            C=c_val,
            solver='lbfgs', 
            max_iter=1000
        ))
    ])

def optimize_pipeline(X, y, n_trials=50):
    def objective(trial):
        pipeline = build_tuned_pipeline(trial)
        
        # We use cross_val_score for speed during optimization
        # Use 'f1_weighted' to handle the class imbalance seen in your CM
        scores = cross_val_score(pipeline, X, y, cv=5, scoring='f1_weighted')
        
        # Stability check: Penalize high variance (std) 
        # to ensure the model isn't just getting lucky
        return scores.mean() - scores.std()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    print("\n--- Optimization Finished ---")
    print(f"Best Score: {study.best_value:.4f}")
    print(f"Best Params: {study.best_params}")
    return study.best_params

def build_dataset(task_diagnosis_frames, tasks, diagnoses, min_tokens=2):
    X, y = [], []
    skipped = 0 
    for task in tasks:
        for diagnosis in diagnoses:
            frames = task_diagnosis_frames[task][diagnosis]
            for stem, df in frames.items():
                tokens = [t.strip() for t in df['token'].astype(str).tolist() if t.strip()]
                if len(tokens) < min_tokens:
                    skipped += 1
                    continue
                X.append(' '.join(tokens))
                y.append(diagnosis)
    print(f"Skipped {skipped} short transcripts")
    return X, y

def run_cv_experiment_with_cm(X, y, k=5):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pipeline = build_pipeline()
    metrics = []
    conf_matrices = []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = [X[i] for i in train_idx], [X[i] for i in test_idx]
        y_train, y_test = [y[i] for i in train_idx], [y[i] for i in test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
        cm = confusion_matrix(y_test, y_pred, labels=TARGET_DIAGNOSES)

        metrics.append((acc, prec, rec, f1))
        conf_matrices.append(cm)

    return np.array(metrics), conf_matrices

def plot_avg_confusion_matrix(conf_matrices, labels, task_name):
    avg_cm = np.mean(conf_matrices, axis=0)
    plt.figure()
    plt.imshow(avg_cm, cmap='Blues')
    plt.title(f"Average Confusion Matrix ({task_name})")
    plt.colorbar()
    plt.xticks(range(len(labels)), labels)
    plt.yticks(range(len(labels)), labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, f"{avg_cm[i, j]:.2f}", ha="center", va="center")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

## Final Analysis

In [ ]:
for task in TARGET_TASKS:
    print(f"\n===== TASK: {task} =====")
    X, y = build_dataset(task_diagnosis_frames, [task], TARGET_DIAGNOSES)
    best_params = optimize_pipeline(X, y, n_trials=30)
    
    if not X:
        print(f"No valid data for {task}")
        continue

    metrics, cms = run_cv_experiment_with_cm(X, y)
    
    # Print Summary
    mean, std = metrics.mean(axis=0), metrics.std(axis=0)
    print(f"Accuracy: {mean[0]:.3f} ± {std[0]:.3f}")
    print(f"F1-score: {mean[3]:.3f} ± {std[3]:.3f}")

    # Visuals
    plot_avg_confusion_matrix(cms, TARGET_DIAGNOSES, task)
    plot_tfidf_comparison(task, task_diagnosis_frames[task], 20)